In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
import os
from tqdm import tqdm

In [ ]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "id": item["id"],
            "premises": item["premises"],
            "conclusion": item["conclusion"],
            "label": item["label"]
        }

In [ ]:
class FOLModel:
    def __init__(
        self,
        model_name: str = "/kaggle/input/models/ductri0981/deepseek-model/transformers/default/3/deepseek_model",
        output_dir: str = "/kaggle/working/fol_model",
        max_length: int = 768,
        local_files_only=True
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=local_files_only
        )

        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.use_cache = False
        self.labels = ["T", "F", "U"]
        self.label_token_ids = {
            label: self.tokenizer(label, add_special_tokens=False).input_ids[0]
            for label in self.labels
        }
        self.labels_mapping = {
            "T": "True",
            "F": "False",
            "U": "Uncertain"
        }

    def predict(self, loader):
        all_results = []
    
        self.model.eval()
        with torch.no_grad():
            for batch in tqdm(loader):
                ids = batch["id"]
                labels = batch["label"]
    
                # ====== tạo input RAW ======
                texts = [
                    f"""You are an expert in logic and reasoning.
Determine whether the conclusion logically follows from the premises.
Do not include explanations, markdown, or extra text.

Labels:
- T: if the conclusion is definitely true from the premises
- F: if the conclusion is definitely false from the premises
- U: if the conclusion cannot be determined to be whether true or false from the premises

Example 1:
Premise: All birds have feathers. A penguin is a bird.
Conclusion: A penguin has feathers.
Label: T

Example 2:
Premise: If the sun is out, Alice goes for a walk. The sun is out.
Conclusion: Alice does not go for a walk.
Label: F

Example 3:
Premise: If a box is blue, it is heavy. Ben has a blue box.
Conclusion: Ben's box contains gold.
Label: U

Now solve the following problem.

Premise: {p}
Conclusion: {c}
Label:
"""
                    for p, c in zip(batch["premises"], batch["conclusion"])
                ]
    
                inputs = self.tokenizer(
                    texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=self.max_length
                ).to(self.model.device)
    
                outputs = self.model(**inputs)
    
                logits = outputs.logits
                last_logits = logits[:, -1, :]  # (B, vocab)
    
                batch_preds = []
    
                for i in range(last_logits.size(0)):
                    scores = {
                        label: last_logits[i, token_id].item()
                        for label, token_id in self.label_token_ids.items()
                    }
    
                    pred = max(scores, key=scores.get)
                    
                    batch_preds.append(pred)
    
                for id_, label, pred in zip(ids, labels, batch_preds):
                    all_results.append({
                        "id": id_,
                        "predict": self.labels_mapping[pred],
                        "label": label
                    })
    
        return all_results

In [ ]:
import json
with open("/kaggle/input/datasets/ductri0981/manual-test/manual_test.json", 'r', encoding="utf-8") as f:
    dataset = json.load(f)
dataset = ManualDataset(dataset)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
)
model = FOLModel()

In [ ]:
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

results_predict = model.predict(loader)
count_correct = 0
for result_predict in results_predict:
    if result_predict["label"] == result_predict["predict"]:
        count_correct += 1
accuracy = count_correct / len(results_predict)

result = {
    "accuracy": f"{accuracy:.4f}",
    "data": results_predict
}

with open("result.json", 'w', encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)